In [ ]:
#| default_exp utils

In [ ]:
from dialoghelper import *

# dialoghelper.utils

In [ ]:
#| export
import os,re,inspect,ast,collections,time,asyncio,json,linecache,importlib,difflib,uuid,builtins,subprocess

from typing import Dict
from tempfile import TemporaryDirectory
from ipykernel_helper import *
from dataclasses import dataclass
from os.path import normpath
from fastcore.xml import to_xml
from fastcore.meta import splice_sig, delegates, delegated

from fastcore.utils import *
from fastcore.xtras import asdict
from fastcore.docments import MarkdownRenderer
from ghapi.all import *
from inspect import currentframe,Parameter,signature
from httpx import AsyncClient, get as xget, post as xpost
from IPython.display import display,Markdown,HTML
from monsterui.all import franken_class_map,apply_classes
from toolslm.xml import *
from fasthtml.common import *
from fasthtml.components import Solveit_input
from urllib.parse import urlencode
from safepyrun import RunPython,find_var,create_python_magic,load_ipython_extension
from pyskills import allow
from pyskills.edit import *

from dialoghelper.core import *
from dialoghelper.core import _msg_edit

In [ ]:
#| export
_lt = import_no_init('fastllm.chat')

In [ ]:
from fastcore import tools
from fastcore.test import *

## Helpers

In [ ]:
#| export
md_cls_d = {
    **{f'h{i}': f'uk-h{i}' for i in range(1,5)},
    'a': "uk-link",
    'blockquote': "uk-blockquote",
    'hr':'uk-divider-icon',
    'table':'uk-table uk-table-sm uk-table-middle uk-table-divider border [&_tr]:divide-x w-[80%] mx-auto',
    'ol': 'uk-list uk-list-decimal space-y-0', 
    'ul': 'uk-list uk-list-bullet space-y-0',
    'p': 'leading-tight',
    'li': 'leading-tight',
    'pre': '', 'pre code': '',
    'code': 'tracking-tight'
}

def add_styles(s:str, cls_map:dict=None):
    "Add solveit styles to `s`"
    return Safe(apply_classes(s, class_map=cls_map or md_cls_d))

In [ ]:
import mistletoe
from fasthtml.common import show

In [ ]:
s = mistletoe.markdown("### hi\n\n- first\n- *second*")
s

'<h3>hi</h3>\n<ul>\n<li>first</li>\n<li><em>second</em></li>\n</ul>\n'

In [ ]:
show(s)

HTML(<h3>hi</h3>
<ul>
<li>first</li>
<li><em>second</em></li>
</ul>
)

In [ ]:
show(add_styles(s))

HTML(<h3 class="uk-h3">hi</h3>
<ul class="uk-list uk-list-bullet space-y-0">
<li class="leading-tight">first</li>
<li class="leading-tight"><em>second</em></li>
</ul>
)

## ast-grep

In [ ]:
#| export
def ast_py(code:str):
    "Get an SgRoot root node for python `code`"
    from ast_grep_py import SgRoot
    return SgRoot(code, "python").root()

In [ ]:
node = ast_py("print('hello world')")
stmt = node.find(pattern="print($A)")
res = stmt.get_match('A')
res.text(),res.range()

("'hello world'",
 Range(start=Pos(line=0, col=6, index=6), end=Pos(line=0, col=19, index=19)))

In [ ]:
#| export
def ast_grep(
    pattern:str, # ast-grep pattern to search, e.g "post($A, data=$B, $$$)"
    path:str=".", # path to recursively search for files
    lang:str="python" # language to search/scan
): # json format from calling `ast-grep --json=compact
    """Use `ast-grep` to find code patterns by AST structure (not text).
    
    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence
    
    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)
    
    Useful for: Refactoring—find all uses of deprecated APIs or changed signatures; Security review—locate SQL queries, file operations, eval calls; Code exploration—understand how libraries are used across codebase; Pattern analysis—find async functions, error handlers, decorators; Better than regex—handles multi-line code, nested structures, respects syntax"""
    cmd = f"ast-grep --pattern '{pattern}' --lang {lang} --json=compact"
    if path != ".": cmd = f"cd {path} && {cmd}"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return json.loads(res.stdout) if res.stdout else res.stderr

The `ast_grep` function calls the `ast-grep` CLI, which is used for searching code based on its structure rather than just text patterns. Unlike regular expressions that match character sequences, `ast-grep` understands the syntax of programming languages and lets you search for code patterns in a way that respects the language's grammar. This means you can find function calls, variable assignments, or other code constructs even when they're formatted differently or have varying amounts of whitespace.

The key advantage is using metavariables (like `$A`, `$B`, `$$$`) as placeholders in your search patterns. When you search for `xpost($A, data=$B, $$$)`, you're asking to find all calls to `xpost` where the first argument can be anything (captured as `$A`), there's a keyword argument `data` with any value (captured as `$B`), and there may be additional arguments after that (the `$$$` matches zero or more remaining arguments). This is much more reliable than trying to write a regex that handles all the variations of how that function might be called.

In the example below, we search for calls to `xpost` in the parent directory and extract both the matched code and the specific values of our metavariables, showing us exactly where and how this function is being used in the codebase.

In [ ]:
res = ast_grep(r"xpost($A, data=$B, $$$)", '..')
[(o['text'],o['metaVariables']['single'],o['file']) for o in res]

[('xpost(url, data=data, headers=headers, timeout=timeout)',
  {'A': {'text': 'url',
    'range': {'byteOffset': {'start': 5196, 'end': 5199},
     'start': {'line': 114, 'column': 30},
     'end': {'line': 114, 'column': 33}}},
   'B': {'text': 'data',
    'range': {'byteOffset': {'start': 5206, 'end': 5210},
     'start': {'line': 114, 'column': 40},
     'end': {'line': 114, 'column': 44}}}},
  'dialoghelper/core.py')]

**Basic Patterns:**
- Match code structure directly: `console.log($ARG)` 
- Metavariables capture parts: `$VAR` (single), `$$$` (multiple)
- Patterns match AST structure, not text - whitespace/formatting doesn't matter

**The Colon Issue:**
- **Don't include `:` in patterns** - it's part of Python's concrete syntax, not the AST structure
- ✅ `def $FUNC($$$)` - matches function definitions
- ❌ `def $FUNC($$$):` - too specific, looking for the colon token itself

**When to use `kind` vs `pattern`:**
- `pattern`: Simple direct matches (`await $EXPR`)
- `kind`: Structural node types (`kind: function_declaration`)

**Critical rule for relational searches:**
Always add `stopBy: end` to `has`/`inside` rules to search the entire subtree:
```yaml
has:
  pattern: await $EXPR
  stopBy: end
```

**Escaping in shell:**
Use `\$VAR` or single quotes when using `--inline-rules` from command line

In [ ]:
#| export
def _ast_replace(
    text:str,
    pattern:str, # AST pattern to match, e.g. 'print($A)'; use $VAR for single nodes, $$$ for multiple
    rewrite:str, # Replacement template with metavariables expanded, e.g. 'log($A)'
    lang:str='python' # Source language (see ast-grep --help for supported languages)
 ):
    """Replace code by AST pattern using ast-grep.

    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence

    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)"""
    res = subprocess.run(
        ['ast-grep', 'run', '--pattern', pattern, '--rewrite', rewrite,
         '--lang', lang, '--update-all', '--stdin'],
        input=text, capture_output=True, text=True)
    if res.returncode: raise ValueError(res.stderr.strip())
    if not res.stdout or res.stdout == text: raise ValueError(f"No matches found for pattern: {pattern}")
    return res.stdout

msg_ast_replace  = _msg_edit (_ast_replace, 'msg_ast_replace')

In [ ]:
_ast_id = await add_msg("print('hello')\nprint('world')\nlog('keep')", msg_type='code')

In [ ]:
print(await msg_ast_replace(_ast_id, 'print($A)', 'logger.info($A)'))

@@ -1,3 +1,3 @@
-print('hello')
-print('world')
+logger.info('hello')
+logger.info('world')
 log('keep')


In [ ]:
print((await read_msg(n=0, id=_ast_id, nums=True))['content'])

     1 │ logger.info('hello')
     2 │ logger.info('world')
     3 │ log('keep')


In [ ]:
await del_msg(_ast_id)

{'status': 'success'}

## Context

In [ ]:
#| export
@delegates(folder2ctx)
async def ctx_folder(
    path:Path='.',  # Path to collect
    types:str|list='py,doc',  # list or comma-separated str of ext types from: py, js, java, c, cpp, rb, r, ex, sh, web, doc, cfg
    out=False, # Include notebook cell outputs?
    raw=True, # Add raw message, or note?
    exts:str|list=None, # list or comma-separated str of exts to include (overrides `types`)
    **kwargs
):
    "Convert folder to XML context and place in a new message"
    if exts: types=None
    res = folder2ctx(path, types=types, out=out, exts=exts, **kwargs)
    if not raw: res = f'```\n{res}\n```'
    return await add_msg(res, msg_type='raw' if raw else 'note')


In [ ]:
# ctx_folder('..', max_total=600, sigs_only=True, exts='py')

In [ ]:
#| export
allow(repo2ctx)

@delegates(repo2ctx)
async def ctx_repo(
    owner:str,  # GitHub repo owner
    repo:str,   # GitHub repo name
    types:str|list='py,doc',  # list or comma-separated str of ext types from: py, js, java, c, cpp, rb, r, ex, sh, web, doc, cfg
    exts:str|list=None, # list or comma-separated str of exts to include (overrides `types`)
    out=False, # Include notebook cell outputs?
    raw=True, # Add raw message, or note?
    **kwargs
):
    "Convert GitHub repo to XML context and place in a new message"
    res = repo2ctx(owner, repo, out=out, types=types, exts=exts, **kwargs)
    if exts: types=None
    if not raw: res = f'```\n{res}\n```'
    return await add_msg(res, msg_type='raw' if raw else 'note')

In [ ]:
#| export
async def ctx_symfile(sym):
    "Add note with filepath and contents for a symbol's source file"
    return await add_msg(sym2file(sym), msg_type='note');


In [ ]:
# ctx_symfile(TemporaryDirectory)

In [ ]:
#| export
@delegates(sym2folderctx)
async def ctx_symfolder(
    sym, # Symbol to get folder context from
    **kwargs):
    "Add raw message with folder context for a symbol's source file location"
    return await add_msg(sym2folderctx(sym, **kwargs), msg_type='raw');


In [ ]:
# ctx_symfolder(folder2ctx)

In [ ]:
#| export
@delegates(sym2folderctx)
async def ctx_sympkg(
    sym, # Symbol to get folder context from
    **kwargs):
    "Add raw message with repo context for a symbol's root package"
    return await add_msg(sym2pkgctx(sym, **kwargs), msg_type='raw');


In [ ]:
# ctx_sympkg(folder2ctx)

## Gists and github

In [ ]:
#| export
@allow
def load_gist(gist_id:str):
    "Retrieve a gist"
    api = GhApi()
    if '/' in gist_id: *_,user,gist_id = gist_id.split('/')
    else: user = None
    return api.gists.get(gist_id, user=user)

In [ ]:
gistid = 'jph00/e7cfd4ded593e8ef6217e78a0131960c'
gist = load_gist(gistid)
gist.html_url

'https://gist.github.com/jph00/e7cfd4ded593e8ef6217e78a0131960c'

In [ ]:
#| export
def gist_file(gist_id:str):
    "Get the first file from a gist"
    gist = load_gist(gist_id)
    return first(gist.files.values())

In [ ]:
gfile = gist_file(gistid)
print(gfile.content[:100]+"…")

"This is a test module which makes some simple tools available."
__all__ = ["hi","whoami"]

testfoo=…


In [ ]:
#| export
def import_string(
    code:str, # Code to import as a module
    name:str  # Name of module to create
):
    with TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / f"{name}.py"
        path.write_text(code)
        # linecache.cache storage allows inspect.getsource() after tmpdir lifetime ends
        linecache.cache[str(path)] = (len(code), None, code.splitlines(keepends=True), str(path))
        spec = importlib.util.spec_from_file_location(name, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
        return module

In [ ]:
def hi(who:str):
    "Say hi to `who`"
    return f"Hello {who}"

def hi2(who):
    "Say hi to `who`"
    return f"Hello {who}"

def hi3(who:str):
    return f"Hello {who}"

bye = "bye"

In [ ]:
assert is_usable_tool(hi)
assert not is_usable_tool(hi2)
assert not is_usable_tool(hi3)
assert not is_usable_tool(bye)

In [ ]:
#| export
def mk_toollist(syms):
    return "\n".join(f"- &`{sym.__name__}`: {sym.__doc__}" for sym in syms if is_usable_tool(sym))

In [ ]:
print(mk_toollist([hi]))

- &`hi`: Say hi to `who`


In [ ]:
#| export
def import_gist(
    gist_id:str, # user/id or just id of gist to import as a module
    mod_name:str=None, # module name to create (taken from gist filename if not passed)
    add_global:bool=True, # add module to caller's globals?
    import_wildcard:bool=False, # import all exported symbols to caller's globals
    create_msg:bool=False # Add a message that lists usable tools
):
    "Import gist directly from string without saving to disk"
    fil = gist_file(gist_id)
    mod_name = mod_name or Path(fil['filename']).stem
    module = import_string(fil['content'], mod_name)
    glbs = currentframe().f_back.f_globals
    if add_global: glbs[mod_name] = module
    syms = getattr(module, '__all__', None)
    if syms is None: syms = [o for o in dir(module) if not o.startswith('_')]
    syms = [getattr(module, nm) for nm in syms]
    if import_wildcard:
        for sym in syms: glbs[sym.__name__] = sym
    if create_msg:
        pref = getattr(module, '__doc__', "Tools added to dialog:")
        asyncio.ensure_future(add_msg(f"{pref}\n\n{mk_toollist(syms)}"))
    return module

In [ ]:
import_gist(gistid)
importtest.testfoo

'testbar'

In [ ]:
import_gist.__doc__

'Import gist directly from string without saving to disk'

In [ ]:
import_gist(gistid, import_wildcard=True)
importtest.testfoo

'testbar'

In [ ]:
hi("Sarah")

'Hello Sarah'

In [ ]:
importtest.__all__

['hi', 'whoami']

In [ ]:
#| export
def update_gist(gist_id:str, content:str):
    "Update the first file in a gist with new content"
    api = GhApi()
    if '/' in gist_id: *_,user,gist_id = gist_id.split('/')
    gist = api.gists.get(gist_id)
    fname = first(gist.files.keys())
    res = api.gists.update(gist_id, files={fname: {'content': content}})
    return res['html_url']

In [ ]:
#| export
def _filter_diff(diff, folder='solveit/', skip_files=('_modidx.py',)):
    "Filter unified diff to only include files under `folder`, skipping `skip_files`"
    sections = re.split(r'(?=^diff --git )', diff, flags=re.MULTILINE)
    return ''.join(s for s in sections
                   if s.startswith(f'diff --git a/{folder}')
                   and not any(f'/{f} ' in s.split('\n')[0] for f in skip_files))

def _reduce_ctx(diff):
    "Keep only diff headers and changed lines (optionally with `n` context lines)"
    lines = diff.splitlines(True)
    return ''.join(l for l in lines if l[0:1] in ('+','-','d','i','@') or not l.strip())

def _fmt_replies(api, owner, repo, num):
    "Format issue/PR comment replies"
    comments = api.issues.list_comments(owner, repo, num)
    if not comments: return ''
    return '\n\n## Replies\n' + '\n\n---\n'.join(f"**@{c.user.login}** ({c.created_at[:10]}):\n{c.body}" for c in comments)

In [ ]:
#| export
@allow
def read_pr(
    pr_number:int|str, # Issue/PR number, or GitHub issue/PR URL
    owner:str='answerdotai', # Owner
    repo:str=None, # Repo
    folder:str='', # For diffs, limit to only to files in `folder`
    replies:bool=False # Include replies
):
    "Fetch a GitHub PR or issue's title, body, optionally replies, and diff (if PR)"
    if '/' in str(pr_number): *_,owner,repo,typ,pr_number = str(pr_number).rstrip('/').split('/')
    pr_number = int(pr_number)
    if folder: folder = f"{folder}/"
    api = GhApi()
    res = None
    try: pr = api.pulls.get(owner, repo, pr_number)
    except:
        try: iss = api.issues.get(owner, repo, pr_number)
        except Exception as e: return f"Err: {e}"
        title,body = iss.title, iss.body or ''
        evts = api.issues.list_events(owner, repo, pr_number)
        sha = first(e.commit_id for e in evts if e.commit_id)
        if not sha: res = f"# {title}\n\n{body}"
        else:
            try: diff = api.repos.get_commit(owner, repo, sha, headers={'Accept': 'application/vnd.github.v3.diff'})
            except Exception as e: res = f"# {title}\n\n{body}\n\n(No diff: {e})"
    else:
        title,body = pr.title, pr.body or ''
        try: diff = api.pulls.get(owner, repo, pr_number, headers={'Accept': 'application/vnd.github.v3.diff'})
        except Exception as e: res = f"# {title}\n\n{body}\n\n(No diff: {e})"
    if res is None:
        diff = _reduce_ctx(_filter_diff(diff, folder=folder))
        res = f"# {title}\n\n{body}\n\n## Diff\n```diff\n{diff}\n```"
    if replies: res += _fmt_replies(api, owner, repo, pr_number)
    return res

## Input

`input` can take a string prompt as normal. OR, you can supply custom UI. For it to work, you must: 

- Wrap in `<solveit-input>` tags (done automatically by the new `input` function defined here)
- Post to `"/input_reply_"` with a value for `user_input`.
- For access keys (shortcuts for buttons), include an `accesskey="y"` in the button, and make sure it is in a form/div with id `'input-request-form'`.

The `show_prompt` demo here (using the custom `InputBtn`) demonstrates this to give a custom yes/no prompt:

In [ ]:
#| export
def InputBtn(txt, value=None, btncls=(), **kw):
    btncls = ' '.join(f'uk-btn-{o}' for o in listify(btncls))
    return Button(txt, type='submit', name='user_input', value=value or txt, cls=f'uk-btn {btncls}', **kw)

def input(prompt='', *args):
    "Solveit customised input to handle fasttag prompts"
    return builtins.input(prompt if isinstance(prompt,str) else Solveit_input(prompt, *args))

def InputForm(*c, **kwargs):
    "Create an `input()` with a `Form` with needed `hx_post` and `id`"
    return input(Form(*c,
        hx_post="/input_reply_", id='input-request-form', **kwargs))

In [ ]:
def show_prompt():
    return InputForm(
        Div('Ship this change now?'),
        Div(cls='flex gap-2')(
            InputBtn('Yes', btncls='primary', accesskey="y"),
            InputBtn('No', btncls='default', accesskey="n"))
    )

In [ ]:
# show_prompt()

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()